# 3.6 Linear regression via gradient descent

Gradient descent reaches the OLS solution by taking repeated downhill steps on squared error.

The notebook fits the same regression method from a hand line to a high-dimensional noisy D5. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

np.random.seed(7)

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []
    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))
    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))
    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))
    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))
    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))
    return rungs

def reg_rmse(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out RMSE."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return float(np.sqrt(mean_squared_error(y_te, preds)))

def linear_baseline(x_tr, y_tr, x_te):
    clf = LinearRegression()
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


## The concept, built once on D1

The lesson formula is $$ \beta_{t+1}=\beta_t-\eta\frac{2}{m}X^\top(X\beta_t-y) $$. The next cell recomputes the exact plan arithmetic before model fitting.

In [ ]:

def linear_regression_via_gradient_descent_method():
    losses = np.array([0.246, 0.096, 0.471], dtype=float)
    raw_sum = float(losses.sum())
    empirical_risk = round(float(raw_sum / len(losses)), 3)
    cost = 0.050
    score = round(empirical_risk + cost, 3)
    alternative = 0.361
    gap = round(alternative - score, 3)
    relative_gap = round(gap / alternative, 3)
    stable_score = round(0.80 * score, 3)
    final_score = min(score, alternative, stable_score)
    return {
        "losses": losses,
        "sum": raw_sum,
        "risk": empirical_risk,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stable": stable_score,
        "final": final_score,
    }

lesson_check = linear_regression_via_gradient_descent_method()
print("losses:", lesson_check["losses"])
print("R_S =", round(lesson_check["sum"], 3), "/ 3 =", round(lesson_check["risk"], 3))
print("score =", round(lesson_check["score"], 3))
print("gap =", round(lesson_check["gap"], 3))
print("relative gap =", round(lesson_check["relative_gap"], 3))
print("stable score =", round(lesson_check["stable"], 3))
assert np.isclose(round(lesson_check["sum"], 3), 0.813)
assert np.isclose(round(lesson_check["risk"], 3), 0.271)
assert np.isclose(round(lesson_check["score"], 3), 0.321)
assert np.isclose(round(lesson_check["gap"], 3), 0.040)
assert np.isclose(round(lesson_check["relative_gap"], 3), 0.111)
assert np.isclose(round(lesson_check["stable"], 3), 0.257)


The exact assertions make the score, gap, and stabilized decision match the lesson.

In [ ]:

def design_with_intercept(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

def normal_equation_fit(X, y, ridge=0.0):
    X_design = design_with_intercept(X)
    gram = X_design.T.dot(X_design)
    penalty = ridge * np.eye(gram.shape[0])
    penalty[0, 0] = 0.0
    beta = np.linalg.pinv(gram + penalty).dot(X_design.T).dot(y)
    return beta

def normal_equation_predict(beta, X):
    return design_with_intercept(X).dot(beta)

def ols_predictor(x_tr, y_tr, x_te):
    beta = normal_equation_fit(x_tr, y_tr)
    return normal_equation_predict(beta, x_te)

def gradient_descent_fit(X, y, lr=0.03, steps=2500):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_design = design_with_intercept(X_scaled)
    beta = np.zeros(X_design.shape[1])
    losses = []
    for step in range(steps):
        residual = X_design.dot(beta) - y
        grad = (2.0 / len(y)) * X_design.T.dot(residual)
        beta = beta - lr * grad
        if step % 50 == 0 or step == steps - 1:
            losses.append(float(np.mean(residual ** 2)))
    return beta, scaler, losses

def gradient_descent_predict(beta, scaler, X):
    X_scaled = scaler.transform(X)
    return design_with_intercept(X_scaled).dot(beta)

def gd_predictor(x_tr, y_tr, x_te):
    beta, scaler, losses = gradient_descent_fit(x_tr, y_tr, lr=0.03, steps=2500)
    return gradient_descent_predict(beta, scaler, x_te)

def run_regression_ladder(kind):
    rows = []
    for rung, (name, X, y) in enumerate(reg_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
        if kind == "ols":
            beta = normal_equation_fit(x_tr, y_tr)
            pred = normal_equation_predict(beta, x_te)
            history = []
        else:
            beta, scaler, history = gradient_descent_fit(x_tr, y_tr)
            pred = gradient_descent_predict(beta, scaler, x_te)
        mse = float(mean_squared_error(y_te, pred))
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "metric": mse,
            "rmse": float(np.sqrt(mse)),
            "x_te": x_te,
            "y_te": y_te,
            "pred": pred,
            "history": history,
        })
    return rows

def plot_regression_panel(ax, X, y, pred, title):
    if X.shape[1] == 1:
        ax.scatter(X[:, 0], y, s=14, alpha=0.6)
        order = np.argsort(X[:, 0])
        ax.plot(X[order, 0], pred[order], color="crimson", linewidth=2)
    else:
        ax.scatter(y, pred, s=14, alpha=0.6)
        low = min(float(np.min(y)), float(np.min(pred)))
        high = max(float(np.max(y)), float(np.max(pred)))
        ax.plot([low, high], [low, high], color="crimson", linewidth=1)
    ax.set_title(title, fontsize=8)
    ax.tick_params(labelsize=7)


## The dataset ladder

The shared `reg_ladder` supplies D1–D5, ending with a real high-dimensional noisy regression stress test.

In [ ]:

rungs = reg_ladder()
for name, X, y in rungs:
    preview = np.round(X[:3, :min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  target range:", (round(float(np.min(y)), 3), round(float(np.max(y)), 3)))
    print("  sample columns:")
    print(preview)


## Run the same regression method across D1–D5

The plan metric is MSE; RMSE is printed only as a readable secondary annotation.

In [ ]:

results = run_regression_ladder("gd")
print("rung | MSE | RMSE | dimensions")
for row in results:
    print(f"D{row['rung']} | {row['metric']:.3f} | {row['rmse']:.3f} | {row['d']}")
helper_rmse = reg_rmse(gd_predictor, rungs[-1][1], rungs[-1][2])
print("D5 helper reg_rmse:", round(helper_rmse, 3))


## Results visualization

The closing figure shows fitted values or residual structure per rung, plus MSE over complexity.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
flat_axes = axes.ravel()
for ax, row in zip(flat_axes[:5], results):
    plot_regression_panel(ax, row["x_te"], row["y_te"], row["pred"], f"D{row['rung']} MSE={row['metric']:.1f}")
flat_axes[5].plot([row["rung"] for row in results], [row["metric"] for row in results], marker="o")
flat_axes[5].set_xlabel("rung")
flat_axes[5].set_ylabel("MSE")
fig.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

For regression the raw MSE can favor a brittle high-flexibility fit; the fix adds the lesson cost and compares on one scale.

In [ ]:

d5 = results[-1]
raw_mse = d5["metric"]
cost = lesson_check["cost"]
fixed_score = raw_mse + cost * raw_mse
stabilized_score = 0.80 * fixed_score
print("D5:", d5["name"])
print("wrong raw MSE:", round(raw_mse, 3))
print("fixed MSE plus scaled cost:", round(fixed_score, 3))
print("stabilized score:", round(stabilized_score, 3))
assert fixed_score > raw_mse
assert stabilized_score < fixed_score


## Evaluate it + Practice

- Metric: MSE; compare against the mean-target baseline.
- Sanity check: residuals should not show a simple missed line on D1/D2.
- Ablation: remove scaling before gradient descent or remove the cost term before selection.
- Failure signals: exploding loss, residual trend, or D5 improvement only on the raw score.

Practice 1: Add a ridge value to the normal equation and rerun D5.

Practice 2: Change the gradient-descent learning rate and plot the loss curve.

Practice 3: Compute the mean-target baseline MSE for every rung.